<a href="https://colab.research.google.com/github/G-Thor/T-715-SPPR/blob/master/ESPnetEZ/TTS_finetune_vctk_dump.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning VITS for Text-to-Speech Synthesis on a New Dataset
In this tutorial, we will guide you through the process of performing text-to-speech (TTS) synthesis by fine-tuning the VITS model on the VCTK dataset. This demo covers data preparation from dump files, model fine-tuning, inference, and evaluation.

## Overview
- Task: Text-to-Speech (TTS)
- Dataset: [VCTK](http://www.udialogue.org/download/cstr-vctk-corpus.html)
- Model: VITS - [espnet/kan-bayashi_libritts_xvector_vits](https://huggingface.co/espnet/kan-bayashi_libritts_xvector_vits)

## License Reminder
Before proceeding, please note that the dataset and model used in this tutorial come with specific licensing terms:
- **VCTK Corpus:** Licensed under the Open Data Commons Attribution License (ODC-By) v1.0.
- **Model:** The pretrained VITS model is under the Creative Commons Attribution 4.0 License.


# Prepare Environment

## Clone ESPnet's Repository

In [ ]:
!git clone https://github.com/espnet/espnet.git

## Install ESPnet and Dependencies

In [ ]:
# NOTE: pip shows imcompatible errors due to preinstalled libraries but you do not need to care
# ESPnet installation
!git clone --depth 5 https://github.com/espnet/espnet.git
!cd espnet && pip install .

!pip install espnet_model_zoo tensorboard

!pip install pyopenjtalk==0.4
!pip install pypinyin==0.44.0
!pip install gdown==4.4.0
!pip install ipywebrtc

# Evaluation related
!git clone --depth 5 https://github.com/shinjiwlab/versa.git
!cd versa && pip install .
!git clone https://github.com/ftshijt/versa_demo_egs.git

import nltk
nltk.download('averaged_perceptron_tagger_eng')

## Import ESPnetEZ

In [ ]:
import espnetez as ez

# Data Preparation

In this tutorial, we will use ESPnet-generated dump files as our inputs. Set up the directory where your processed dump folder is stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ADJUST THIS PATH TO MATCH THE PATH TO YOUR DUMP DIRECTORY
PROJECT_ROOT = f"/content/drive/MyDrive/your/data"
DUMP_DIR = f"{PROJECT_ROOT}/dump"
data_info = {
    "speech": ["wav.scp", "sound"],
    "text": ["text", "text"],
}

# Fine-Tuning

## Download Pretrained VITS Model
We'll use ESPnet's model zoo to download the [pretrained VITS model from the LibriTTS corpus](https://huggingface.co/espnet/kan-bayashi_libritts_xvector_vits).


In [ ]:
from espnet_model_zoo.downloader import ModelDownloader

PRETRAIN_MODEL = "espnet/kan-bayashi_libritts_xvector_vits"
#PRETRAIN_MODEL = "espnet/kan-bayashi_vctk_xvector_tacotron2"
d = ModelDownloader()
pretrain_downloaded = d.download_and_unpack(PRETRAIN_MODEL)

## Configure Fine-Tuning

Load the pretrained model's configuration and set it up for fine-tuning.

In [ ]:
TASK = "gan_tts"

pretrain_config = ez.config.from_yaml(TASK, pretrain_downloaded["train_config"])

# Update the configuration with the downloaded model file path
pretrain_config["model_file"] = pretrain_downloaded["model_file"]


# Modify configuration for fine-tuning
finetune_config = pretrain_config.copy()
finetune_config["init_param"] = [pretrain_downloaded["model_file"]]
finetune_config["batch_size"] = 20
finetune_config["num_workers"] = 0
finetune_config["max_epoch"] = 20
finetune_config["batch_bins"] = 500000
finetune_config["num_iters_per_epoch"] = None
finetune_config["generator_first"] = True

# Lower LR for fine-tuning (pretrained uses 2e-4)
finetune_config["optim_conf"] = {
    "lr": 1.0e-5,
    "betas": [0.8, 0.99],
    "eps": 1.0e-9,
    "weight_decay": 0.0,
}
finetune_config["optim2_conf"] = {
    "lr": 1.0e-5,
    "betas": [0.8, 0.99],
    "eps": 1.0e-9,
    "weight_decay": 0.0,
}

# Disable distributed training
finetune_config["distributed"] = False
finetune_config["multiprocessing_distributed"] = False
finetune_config["dist_world_size"] = None
finetune_config["dist_rank"] = None
finetune_config["local_rank"] = None
finetune_config["dist_master_addr"] = None
finetune_config["dist_master_port"] = None
finetune_config["dist_launcher"] = None

# # Optionally freeze text encoder to prevent forgetting phoneme knowledge
# finetune_config["freeze_param"] = [
#     "tts.generator.text_encoder",
# ]

In [ ]:
# # UNCOMMENT AND RUN TO REMOVE THE EXP DIRECTORY AND ALL ITS CONTENTS
# # Only use in case you started a training run with the wrong configuration
# import shutil
# import os

# # Define the experiment directory path
# EXP_DIR = "./exp/finetune_gan_tts_vctk" # Use the same EXP_DIR as defined later

# # Check if the directory exists and remove it if it does
# if os.path.exists(EXP_DIR):
#     shutil.rmtree(EXP_DIR)
#     print(f"Removed existing experiment directory: {EXP_DIR}")
# else:
#     print(f"Experiment directory not found: {EXP_DIR}")

# # Define the stats directory path (if it also needs to be cleaned)
# STATS_DIR = "./exp/stats_vctk" # Use the same STATS_DIR as defined later
# if os.path.exists(STATS_DIR):
#     shutil.rmtree(STATS_DIR)
#     print(f"Removed existing stats directory: {STATS_DIR}")
# else:
#     print(f"Stats directory not found: {STATS_DIR}")

## Initialize Trainer

Define the trainer for the fine-tuning process.

In [ ]:
DATASET_NAME = "vctk"
EXP_DIR = f"./exp/finetune_{TASK}_{DATASET_NAME}"
STATS_DIR = f"./exp/stats_{DATASET_NAME}"
ngpu = 1

trainer = ez.Trainer(
    task=TASK,
    train_config=finetune_config,
    train_dump_dir=f"{DUMP_DIR}/raw/my_training_set",
    valid_dump_dir=f"{DUMP_DIR}/raw/my_dev_set",
    data_info=data_info,
    output_dir=EXP_DIR,
    stats_dir=STATS_DIR,
    ngpu=ngpu,
)

# Add the xvector paths to the configuration
trainer.train_config.train_data_path_and_name_and_type += [
    [f"{DUMP_DIR}/xvector/my_training_set/xvector.scp", "spembs", "kaldi_ark"],
]
trainer.train_config.valid_data_path_and_name_and_type += [
    [f"{DUMP_DIR}/xvector/my_dev_set/xvector.scp", "spembs", "kaldi_ark"],
]

### Adjust `wav.scp` Paths

The `wav.scp` files generated for ESPnet might contain relative paths. Since the data is located on Google Drive, we need to ensure that the paths within these files are absolute, pointing directly to the audio files on the mounted drive. The `LibsndfileError` indicates that `soundfile` is failing to open the audio files because it's looking for them in the wrong place.

In [ ]:
import os

def fix_kaldi_scp_paths(scp_file_path, base_path_for_relatives):
    """
    Reads an .scp file, prepends a base_path_for_relatives to all relative paths, and writes it back.
    """
    print(f"Fixing paths in: {scp_file_path}")
    fixed_lines = []
    try:
        with open(scp_file_path, 'r') as f:
            for line in f:
                parts = line.strip().split(maxsplit=1)
                if len(parts) == 2:
                    audio_id, path_entry = parts
                    # Split path_entry by ':' to handle Kaldi's ark:offset format
                    path_to_file = path_entry.split(':', 1)[0]
                    offset_part = ':' + path_entry.split(':', 1)[1] if ':' in path_entry else ''

                    # Check if the path is already absolute, if not, prepend the prefix
                    if not os.path.isabs(path_to_file) and not path_to_file.startswith('/content/drive'):
                        fixed_path_to_file = os.path.join(base_path_for_relatives, path_to_file)
                        fixed_lines.append(f"{audio_id} {fixed_path_to_file}{offset_part}")
                    else:
                        fixed_lines.append(line.strip())
                else:
                    fixed_lines.append(line.strip())

        with open(scp_file_path, 'w') as f:
            for line in fixed_lines:
                f.write(f"{line}\n")
        print(f"Successfully fixed paths in {scp_file_path}")
    except FileNotFoundError:
        print(f"Warning: {scp_file_path} not found. Skipping path fixing for this file.")



In [ ]:
# For wav.scp files, the paths within them are relative to the 'Module 2' folder.
WAV_FILES_BASE_PATH = os.path.dirname(DUMP_DIR) # Should be /content/drive/path/to/parent/of/dumpdir
fix_kaldi_scp_paths(f"{DUMP_DIR}/raw/my_training_set/wav.scp", WAV_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/raw/my_dev_set/wav.scp", WAV_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/raw/my_eval_set/wav.scp", WAV_FILES_BASE_PATH)

# For spk_embed.scp files, the paths within them are relative to the DUMP_DIR.
XVECTOR_FILES_BASE_PATH = os.path.dirname(DUMP_DIR) # This should be /content/drive/path/to/parent/of/dumpdir
fix_kaldi_scp_paths(f"{DUMP_DIR}/xvector/my_training_set/xvector.scp", XVECTOR_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/xvector/my_dev_set/xvector.scp", XVECTOR_FILES_BASE_PATH)
fix_kaldi_scp_paths(f"{DUMP_DIR}/xvector/my_eval_set/xvector.scp", XVECTOR_FILES_BASE_PATH)

## Collect Statistics

Before training, we need to collect data statistics (e.g., normalization stats).

In [ ]:
# Temporarily set to None, as we need to collect stats first
trainer.train_config.normalize = None
trainer.train_config.pitch_normalize = None
trainer.train_config.energy_normalize = None

# Collect stats
trainer.collect_stats()

# Restore normalization configs with collected stats
trainer.train_config.write_collected_feats = False
if finetune_config["normalize"] is not None:
    trainer.train_config.normalize = finetune_config["normalize"]
    trainer.train_config.normalize_conf["stats_file"] = (
        f"{STATS_DIR}/my_training_set/feats_stats.npz"
    )
if finetune_config["pitch_normalize"] is not None:
    trainer.train_config.pitch_normalize = finetune_config["pitch_normalize"]
    trainer.train_config.pitch_normalize_conf["stats_file"] = (
        f"{STATS_DIR}/my_training_set/pitch_stats.npz"
    )
if finetune_config["energy_normalize"] is not None:
    trainer.train_config.energy_normalize = finetune_config["energy_normalize"]
    trainer.train_config.energy_normalize_conf["stats_file"] = (
        f"{STATS_DIR}/my_training_set/energy_stats.npz"
    )

## Start Training

Now, let's start the fine-tuning process.

In [ ]:
import torch, functools
torch.load = functools.partial(torch.load, weights_only=False)

In [55]:
trainer.train()

# Inference

When training is done, we can use the inference API to synthesize audio from the test set.

In [ ]:
from espnet2.bin.tts_inference import inference


for i in range(1, 20):
    ckpt_name = f"checkpoint_{i}"
    inference_folder = f"{PROJECT_ROOT}/{EXP_DIR}/inference_{ckpt_name}"
    model_file = f"{EXP_DIR}/{ckpt_name}.pth"
    inference(
        output_dir=inference_folder,
        batch_size=1,
        dtype="float32",
        ngpu=1,
        seed=0,
        num_workers=1,
        log_level="INFO",
        data_path_and_name_and_type=[
            (f"{DUMP_DIR}/raw/my_eval_set/text", "text", "text"),
            (f"{DUMP_DIR}/raw/my_eval_set/wav.scp", "speech", "sound"),
            (f"{DUMP_DIR}/xvector/my_eval_set/xvector.scp", "spembs", "kaldi_ark"),
        ],
        key_file=None,
        train_config=f"{EXP_DIR}/config.yaml",
        model_file=model_file,
        model_tag=None,
        threshold=0.5,
        minlenratio=0.0,
        maxlenratio=10.0,
        use_teacher_forcing=False,
        use_att_constraint=False,
        backward_window=1,
        forward_window=3,
        speed_control_alpha=1.0,
        noise_scale=0.667,
        noise_scale_dur=0.8,
        always_fix_seed=False,
        allow_variable_data_keys=False,
        vocoder_config=None,
        vocoder_file=None,
        vocoder_tag=None,
    )


## Inference with Pre-trained Model on Fine-tuning Eval Set

To compare the performance, we will now run inference using the original pre-trained VITS model on the same evaluation dataset used for fine-tuning.

In [ ]:
from espnet2.bin.tts_inference import inference

pretrain_inference_folder = f"{PROJECT_ROOT}/{EXP_DIR}/inference_pretrain_eval"

inference(
    output_dir=pretrain_inference_folder,
    batch_size=1,
    dtype="float32",
    ngpu=1,
    seed=0,
    num_workers=1,
    log_level="INFO",
    data_path_and_name_and_type=[
        (f"{DUMP_DIR}/raw/my_eval_set/text", "text", "text"),
        (f"{DUMP_DIR}/raw/my_eval_set/wav.scp", "speech", "sound"),
        (f"{DUMP_DIR}/xvector/my_eval_set/xvector.scp", "spembs", "kaldi_ark"),
    ],
    key_file=None,
    train_config=pretrain_downloaded["train_config"],
    model_file=pretrain_downloaded["model_file"],
    model_tag=None,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=10.0,
    use_teacher_forcing=False,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    speed_control_alpha=1.0,
    noise_scale=0.667,
    noise_scale_dur=0.8,
    always_fix_seed=False,
    allow_variable_data_keys=False,
    vocoder_config=None,
    vocoder_file=None,
    vocoder_tag=None,
)

# References

[1] S. Someki, K. Choi, S. Arora, W. Chen, S. Cornell, J. Han, Y. Peng, J. Shi, V. Srivastav, and S. Watanabe, “ESPnet-EZ: Python-only ESPnet for Easy Fine-tuning and Integration,” *arXiv preprint* arXiv:2409.09506, 2024.

[2] C. Veaux, J. Yamagishi, and K. MacDonald, “CSTR VCTK Corpus: English Multi-speaker Corpus for CSTR Voice Cloning Toolkit,” University of Edinburgh, The Centre for Speech Technology Research (CSTR), 2017. [Sound]. https://doi.org/10.7488/ds/1994.

[3] J. Shi, H. Shim, J. Tian, S. Arora, H. Wu, D. Petermann, J. Q. Yip, Y. Zhang, Y. Tang, W. Zhang, D. S. Alharthi, Y. Huang, K. Saito, J. Han, Y. Zhao, C. Donahue, and S. Watanabe, “VERSA: A Versatile Evaluation Toolkit for Speech, Audio, and Music,” arXiv preprint arXiv:2412.17667, 2024.